In [ ]:
from google.colab import drive
import sqlite3
import pandas as pd

# Montado del Drive
drive.mount('/content/drive')

# Config. SQL
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# Conectar BD
ruta = '/content/drive/MyDrive/Hitos_Skillnest/Hito1/34_SuperTienda_Espanol.db'
conn = sqlite3.connect(ruta)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
# Cargar tabla principal a limpiar
df_det_pedidos = pd.read_sql("SELECT * FROM Detalle_Pedido", conn)

# Cargar las tablas maestras solo para la validación final
df_pedidos = pd.read_sql("SELECT * FROM Pedidos", conn)
df_productos = pd.read_sql("SELECT * FROM Productos", conn)

display(df_det_pedidos.head())

DatabaseError: Execution failed on sql 'SELECT * FROM Detalle_Pedido': no such table: Detalle_Pedido

In [ ]:
print("--- 1. ANÁLISIS DE DATOS NULOS O FALTANTES ---")
print(df_det_pedidos.isnull().sum())

print("\n--- 2. DISTRIBUCIÓN DE VOLUMEN (CANTIDADES) ---")
print(df_det_pedidos['Cantidad'].value_counts().sort_index())

--- 1. ANÁLISIS DE DATOS FALTANTES ---
ID_Detalle     0
ID_Pedido      0
ID_Producto    0
Cantidad       0
Descuento      0
Ventas         0
Ganancia       0
Prioridad      0
dtype: int64

--- 2. DISTRIBUCIÓN DE VOLUMEN (CANTIDADES) ---
Cantidad
1    374
2    365
3    417
4    381
5    383
6    362
7    370
8    374
Name: count, dtype: int64


In [ ]:
df_det_pedidos = df_det_pedidos.drop(columns=['ID_Detalle'])

NameError: name 'df_det_pedidos' is not defined

In [ ]:
# 1. Calcular IQR para Ganancias y Ventas
Q1_g = df_det_pedidos['Ganancia'].quantile(0.25)
Q3_g = df_det_pedidos['Ganancia'].quantile(0.75)
IQR_g = Q3_g - Q1_g
limite_inf_g = Q1_g - 1.5 * IQR_g
limite_sup_g = Q3_g + 1.5 * IQR_g

Q3_v = df_det_pedidos['Ventas'].quantile(0.75)
limite_sup_v = Q3_v + 1.5 * (Q3_v - df_det_pedidos['Ventas'].quantile(0.25))

# 2. Crear las variables calculadas por defecto
df_det_pedidos['Etiqueta_Ganancia'] = 'Normal'
df_det_pedidos['Etiqueta_Ventas'] = 'Normal'

# 3. Aplicar las etiquetas a los outliers detectados sean positivos o negativos.
df_det_pedidos.loc[df_det_pedidos['Ganancia'] > limite_sup_g, 'Etiqueta_Ganancia'] = 'Ganancia Atípica'
df_det_pedidos.loc[df_det_pedidos['Ganancia'] < limite_inf_g, 'Etiqueta_Ganancia'] = 'Pérdida Crítica'
df_det_pedidos.loc[df_det_pedidos['Ventas'] > limite_sup_v, 'Etiqueta_Ventas'] = 'Venta Extraordinaria'

print("Variables calculadas agregadas exitosamente.")

Variables calculadas agregadas exitosamente.


In [ ]:
# 4. Verificación de duplicados
print(df_det_pedidos.duplicated().sum())

In [ ]:
# 5. Buscar por si hay IDs fantasmas cruzando con las tablas maestras
pedidos_fantasma = df_det_pedidos[~df_det_pedidos['ID_Pedido'].isin(df_pedidos['ID_Pedido'])]
productos_fantasma = df_det_pedidos[~df_det_pedidos['ID_Producto'].isin(df_productos['ID_Producto'])]

print("--- RESULTADOS DE LA VALIDACIÓN DE INTEGRIDAD ---")
print(f"Fallas en Pedidos: {len(pedidos_fantasma)} registros huérfanos.")
print(f"Fallas en Productos: {len(productos_fantasma)} registros huérfanos.")

--- RESULTADOS DE LA VALIDACIÓN DE INTEGRIDAD ---
Fallas en Pedidos: 0 registros huérfanos.
Fallas en Productos: 0 registros huérfanos.


In [ ]:
# Creación de métrica financiera, variable calculada numérica.
df_det_pedidos['Margen_Rentabilidad_%'] = round((df_det_pedidos['Ganancia'] / df_det_pedidos['Ventas']) * 100, 2)

Enfoque de clientes

Variables a calcular:

Cuales son los segmentos que mas pedidos realizan, Cuales son las categorias mas pedidas por cada segmento, cuales son los segmentos que producen mayor inversion, los canales preferidos mas utilizados en general, los canales preferidos mas utilizado por cada segmento, la media del puntaje de fidelidad, posible relacion entre segmentos y puntos de fidelidad.


In [ ]:
df_det_pedidos